# HTTP endpoint protection and replay

Install Deno and register its TypeScript kernel with `deno jupyter --install`. Open this file
in a Jupyter front end such as JupyterLab or VS Code, select **Deno**, then Run All in a fresh kernel.
The cells import the exact published verifier **2.0.0**.

This demonstration uses a real loopback HTTP listener. Its logical HTTPS audience represents TLS
termination at a host boundary. Verification returns facts; live revocation and authorization belong
to the host. The in-memory replay store lasts for this kernel only.


## Load the published verifier

Check the release and the backend Deno selects.


In [ ]:
import { Buffer } from "node:buffer";
import { BACKEND_ID, assembleCompact, checkEnvelope, grantSigningInput, jsonDecode, jwkFromPublicKey, proofSigningInput, strUtf8, thumbprint, trying, untrustedKeyLocator, uriNormalize, utf8Str, type EnvelopeFacts, type SigningInput, type Tagged, type TrustedIssuer } from "npm:@bounded-authority-protocol/verifier@2.0.0";
import { generateKeyPairSync, randomUUID, sign, type KeyObject } from "node:crypto";
import { createServer, type IncomingMessage, type ServerResponse } from "node:http";
import verifierPackage from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
import assert from "node:assert/strict";
import type { AddressInfo } from "node:net";
assert.equal(verifierPackage.version, "2.0.0");
assert.equal(BACKEND_ID, "portable");
console.log(JSON.stringify({ version: verifierPackage.version, backend: BACKEND_ID }));


{"version":"2.0.0","backend":"portable"}


## Set expected host context

The audience and issuer are host configuration, independent of request credentials.


In [ ]:
// ---- The resource server ----------------------------------------------------------------------

// This server's identifier, which its grants name as their audience, and the origin its clients
// address. TLS terminates in front of the server, so requests arrive here as plain HTTP.
const AUDIENCE = "https://notes.example";
const ISSUER = "https://issuer.example";
const CLOCK_SKEW = 60; // seconds, at most 60
const PROOF_MAX_AGE = 120; // seconds, 1 to 300

// The issuer keys this server enrolled, by kid. A key never comes from the request.
const trustedIssuers = new Map<string, TrustedIssuer>();
assert.equal(trustedIssuers.size, 0); assert.ok(PROOF_MAX_AGE >= 1 && PROOF_MAX_AGE <= 300);
console.log(JSON.stringify({ enrolledKeys: trustedIssuers.size, proofMaxAge: PROOF_MAX_AGE }));


{"enrolledKeys":0,"proofMaxAge":120}


## Define replay reservation

An atomic host store must refuse a second use through the grant expiry plus clock skew. This demonstration uses process memory.


In [ ]:
// Reserve each invocation of a grant once, through the validity of every proof under that grant.
// until is an absolute expiry in integer Unix seconds. Only the store reads its clock.
// Keep the entry at equality; purge only after until. A shared store must do both atomically.
interface ReplayStore { reserve(key: string, until: number): boolean }
function memoryReplayStore(clock: () => number): ReplayStore {
  const reserved = new Map<string, number>();
  return {
    reserve(key, until) {
      const now = clock();
      for (const [k, expiry] of reserved) if (expiry < now) reserved.delete(k);
      if (reserved.has(key)) return false;
      reserved.set(key, until);
      return true;
    },
  };
}
const replay = memoryReplayStore(() => Math.floor(Date.now() / 1000));
assert.equal(typeof replay.reserve, "function");
console.log(JSON.stringify({ reserve: typeof replay.reserve }));


{"reserve":"function"}


## Define bounded request verification

Read a bounded body, look up only enrolled keys, verify received request context, then reserve the invocation before host policy.


In [ ]:
const header = (req: IncomingMessage, name: string): string | undefined => {
  const value = req.headers[name];
  return typeof value === "string" && value !== "" ? value : undefined;
};

// Cap the body while reading: the envelope is not authenticated yet, so an unbounded read would
// let any caller who copies a grant hold this server's memory. Pick your application's limit; the
// protocol's own bounds stop at 64 KiB of JSON.
const MAX_BODY_BYTES = 16_384;

async function readBody(req: IncomingMessage): Promise<Uint8Array | undefined> {
  const chunks: Buffer[] = [];
  let size = 0;
  for await (const chunk of req) {
    size += (chunk as Buffer).length;
    if (size > MAX_BODY_BYTES) return undefined;
    chunks.push(chunk as Buffer);
  }
  return Buffer.concat(chunks);
}

type Handler = (facts: EnvelopeFacts, args: Tagged, res: ServerResponse) => void;

// Middleware for one operation. Every refusal is the same 401, with no detail for the caller.
function requireEnvelope(operation: string, handler: Handler) {
  return async (req: IncomingMessage, res: ServerResponse): Promise<void> => {
    const refuse = (): void => void res.writeHead(401).end();
    const grant = header(req, "bap-grant");
    const proof = header(req, "bap-proof");
    const invocationId = header(req, "bap-invocation");
    if (grant === undefined || proof === undefined || invocationId === undefined) return refuse();

    // The grant's kid is a lookup hint read from its header; the signature check binds the key.
    const locator = untrustedKeyLocator(strUtf8(grant));
    const trustedIssuer = locator.ok ? trustedIssuers.get(locator.value.keyId) : undefined;
    if (trustedIssuer === undefined) return refuse();

    // The expected request comes from this server's own request handling, never from the proof:
    // the method and path it received, and the arguments it parsed (strict JSON, types kept).
    const target = uriNormalize(strUtf8(AUDIENCE + (req.url ?? "/"))); // a query string refuses
    const body = await readBody(req);
    if (body === undefined) return refuse();
    const args = trying(() => jsonDecode(body));
    if (!target.ok || !args.ok) return refuse();

    const now = Math.floor(Date.now() / 1000); // seconds, never Date.now() milliseconds
    const result = checkEnvelope(strUtf8(grant), strUtf8(proof), {
      trustedIssuer, issuer: ISSUER, audience: AUDIENCE, method: req.method ?? "",
      targetUri: utf8Str(target.value), invocationId, operation, castArguments: args.value,
      evaluationTime: now, clockSkew: CLOCK_SKEW, proofMaxAge: PROOF_MAX_AGE, nonce: { kind: "not_required" },
    });
    if (!result.ok) return refuse();

    // A fresh proof can reuse this invocation. Retain it through the grant's expiry plus skew.
    const facts = result.value;
    const key = JSON.stringify([facts.issuer, facts.grantId, facts.invocationId]);
    if (!replay.reserve(key, facts.grantExpiresAt + CLOCK_SKEW)) return refuse();

    // Revocation and your own policy come next. The facts say what was proven, not what to allow.
    handler(facts, args.value, res);
  };
}
assert.ok(MAX_BODY_BYTES <= 65_536); assert.equal(typeof requireEnvelope, "function");
console.log(JSON.stringify({ bodyLimit: MAX_BODY_BYTES, verifier: typeof requireEnvelope }));


{"bodyLimit":16384,"verifier":"function"}


## Create the resource endpoint

The handler runs only after verification and reservation. It still needs the production host policy.


In [ ]:
const createNote = requireEnvelope("notes.create", (facts, _args, res) => {
  res.writeHead(201, { "content-type": "application/json" }).end(JSON.stringify({ grant: facts.grantId }));
});
const server = createServer((req, res) => {
  if (req.method === "POST" && req.url === "/notebooks/team/notes") {
    createNote(req, res).catch(() => res.writeHead(500).end());
    return;
  }
  res.writeHead(404).end();
});
assert.equal(server.listening, false);
console.log(JSON.stringify({ listening: server.listening }));


{"listening":false}


## Issue a selector-bound grant

Enroll a fresh demonstration issuer and grant only the team notebook selector.


In [ ]:
, for this demonstration -----------------------------------------

const publicKeyBytes = (key: KeyObject): Uint8Array => Buffer.from(key.export({ format: "jwk" }).x!, "base64url");
const signingBytes = (input: SigningInput): Uint8Array =>
  Buffer.concat([input.protectedSegment, Buffer.from("."), input.payloadSegment]);
const issuer = generateKeyPairSync("ed25519");
const holder = generateKeyPairSync("ed25519");
trustedIssuers.set("issuer-2026-11", { keyId: "issuer-2026-11", publicKey: publicKeyBytes(issuer.publicKey) });

// The issuer lets the holder add notes to the "team" notebook for one hour.
const issuedAt = Math.floor(Date.now() / 1000);
const grantInput = grantSigningInput({
  keyId: "issuer-2026-11", issuer: ISSUER, grantId: "grant-notes-1", audiences: [AUDIENCE],
  issuedAt, notBefore: issuedAt, expiresAt: issuedAt + 3600,
  holderThumbprint: thumbprint(jwkFromPublicKey(publicKeyBytes(holder.publicKey))),
  operations: [{
    name: "notes.create",
    selectors: [{ kind: "equals", path: ["notebook"], value: { t: "string", v: strUtf8("team") } }],
  }],
});
assert.ok(grantInput.ok);
const signedGrant = assembleCompact(grantInput.value, sign(null, signingBytes(grantInput.value), issuer.privateKey));
assert.ok(signedGrant.ok);
const grant = signedGrant.value;


console.log(JSON.stringify({ ok: signedGrant.ok, enrolledKeys: trustedIssuers.size }));


{"ok":true,"enrolledKeys":1}


## Prepare exact requests

Each request signs the parsed arguments it will actually send.


In [ ]:
// The holder signs one proof per call, over the exact arguments it sends.
interface Call { headers: Record<string, string>; body: string }
function prepare(body: string): Call {
  const invocationId = randomUUID(); // a lowercase RFC 4122 UUID, new for every call
  const proofInput = proofSigningInput({
    holderPublicKey: publicKeyBytes(holder.publicKey), proofId: randomUUID(), method: "POST",
    targetUri: `${AUDIENCE}/notebooks/team/notes`, issuedAt: Math.floor(Date.now() / 1000), invocationId,
    operation: "notes.create", grantCompact: grant, castArguments: jsonDecode(strUtf8(body)),
  });
  assert.ok(proofInput.ok);
  const proof = assembleCompact(proofInput.value, sign(null, signingBytes(proofInput.value), holder.privateKey));
  assert.ok(proof.ok);
  return {
    headers: {
      "content-type": "application/json", "bap-grant": utf8Str(grant),
      "bap-proof": utf8Str(proof.value), "bap-invocation": invocationId,
    },
    body,
  };
}


console.log(JSON.stringify({ prepare: typeof prepare }));


{"prepare":"function"}


## Start the real listener

Select a free loopback port. The port never appears in saved output.


In [ ]:
await new Promise<void>((resolve) => server.listen(0, "127.0.0.1", resolve));
const origin = `http://127.0.0.1:${(server.address() as AddressInfo).port}`;
const send = async (call: Call): Promise<number> =>
  (await fetch(`${origin}/notebooks/team/notes`, { method: "POST", headers: call.headers, body: call.body })).status;

assert.equal(server.listening, true);
console.log(JSON.stringify({ listening: server.listening }));


{"listening":true}


## Send the signed request

The first use reaches the handler.


In [ ]:
const standup = prepare('{"notebook":"team","title":"Standup"}');
const acceptedStatus = await send(standup);
assert.equal(acceptedStatus, 201);
console.log(JSON.stringify({ status: acceptedStatus }));


{"status":201}


## Refuse replay

Sending the same invocation again must fail before another effect.


In [ ]:
const replayStatus = await send(standup);
assert.equal(replayStatus, 401);
console.log(JSON.stringify({ status: replayStatus }));


{"status":401}


## Refuse changed arguments

Changing the body invalidates the request binding.


In [ ]:
const tamperStatus = await send({ ...standup, body: '{"notebook":"team","title":"Edited"}' });
assert.equal(tamperStatus, 401);
console.log(JSON.stringify({ status: tamperStatus }));


{"status":401}


## Refuse a selector outside the grant

A newly signed request still has to satisfy the grant selector.


In [ ]:
const selectorStatus = await send(prepare('{"notebook":"drafts","title":"Ideas"}'));
assert.equal(selectorStatus, 401);
console.log(JSON.stringify({ status: selectorStatus }));


{"status":401}


## Refuse missing credentials

The endpoint applies the same closed refusal to an unauthenticated call.


In [ ]:
const missingStatus = await send({ ...prepare('{"notebook":"team","title":"Retro"}'), headers: {} });
assert.equal(missingStatus, 401);
console.log(JSON.stringify({ status: missingStatus }));


{"status":401}


## Enforce the body cap

A signed body can still exceed the host limit.


In [ ]:
const bodyLimitStatus = await send(prepare(`{"notebook":"team","title":"Big","lines":${JSON.stringify(Array(3).fill("x".repeat(8000)))}}`));
assert.equal(bodyLimitStatus, 401);
console.log(JSON.stringify({ status: bodyLimitStatus }));


{"status":401}


## Accept a fresh invocation

Refusals do not prevent an independent, valid request.


In [ ]:
const freshStatus = await send(prepare('{"notebook":"team","title":"Retro"}'));
assert.equal(freshStatus, 201);
console.log(JSON.stringify({ status: freshStatus }));


{"status":201}


## Close the listener

Release every listener and connection before finishing. Kernel shutdown also releases them if a prior assertion fails.


In [ ]:
server.closeAllConnections();
await new Promise<void>((resolve, reject) => server.close((error) => error ? reject(error) : resolve()));
assert.equal(server.listening, false);
console.log(JSON.stringify({ listening: server.listening }));


{"listening":false}


## Finish

Shut down the kernel to discard demonstration keys and replay state.
Continue with the [HTTP guide](../guides/http-server.md).
